In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# Step 1: Data Collection & Pre-processing (10 Marks)
# ---------------------------------------------------------
# Load the dataset
df = pd.read_csv('Loan_Dataset.csv')

# Drop the unique identifier as it has no predictive value
df = df.drop('Applicant_ID', axis=1)

# Handle Missing Values (Good practice, even if your current sample looks clean)
# Fill categorical columns with the mode
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])
    
# Fill numerical columns with the median
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

# ---------------------------------------------------------
# Step 2: Innovation / Creativity in Approach (5 Marks)
# ---------------------------------------------------------
# Feature Engineering: Calculate the Debt-to-Income (DTI) Ratio
# DTI is a crucial metric used by real banks to assess risk
df['Monthly_Income'] = df['Annual_Income'] / 12
df['DTI_Ratio'] = df['Monthly_Expenses'] / df['Monthly_Income']
df = df.drop('Monthly_Income', axis=1) # Drop temporary column

# ---------------------------------------------------------
# Step 3: Encoding Categorical Variables
# ---------------------------------------------------------
# 3a. Label Encoding for binary categories (2 options)
binary_cols = ['Gender', 'Marital_Status', 'Education', 'Loan_Type', 'Co-Applicant']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 3b. One-Hot Encoding for nominal categories (3+ options)
# drop_first=True prevents the "dummy variable trap" (multicollinearity)
nominal_cols = ['Employment_Status', 'Occupation_Type', 'Residential_Status', 'City/Town', 'Loan_Purpose']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# ---------------------------------------------------------
# Step 4: Model Development & Implementation (12 Marks)
# ---------------------------------------------------------
# Separate features (X) and target variable (y)
X = df.drop('Loan_Approval_Status', axis=1)
y = df['Loan_Approval_Status']

# Train/Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Feature Scaling: Standardize the data so large values (Income) 
# don't overpower small values (Interest Rate)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predict on the hidden test set
y_pred = rf_model.predict(X_test_scaled)

# ---------------------------------------------------------
# Step 5: Performance Evaluation (8 Marks)
# ---------------------------------------------------------
print("--- Random Forest Model Performance ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

--- Random Forest Model Performance ---
Accuracy Score: 0.8538

Confusion Matrix:
[[3919 1611]
 [ 669 9401]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.71      0.77      5530
           1       0.85      0.93      0.89     10070

    accuracy                           0.85     15600
   macro avg       0.85      0.82      0.83     15600
weighted avg       0.85      0.85      0.85     15600



In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the three models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Loop through each model, train it, and print the evaluation
for name, model in models.items():
    print(f"==================================================")
    print(f"--- {name} ---")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions on the test set
    y_pred = model.predict(X_test_scaled)
    
    # Print the metrics
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print(classification_report(y_test, y_pred, zero_division=0))
print(f"==================================================")

--- Logistic Regression ---
Accuracy: 0.8531

              precision    recall  f1-score   support

           0       0.85      0.71      0.77      5530
           1       0.85      0.93      0.89     10070

    accuracy                           0.85     15600
   macro avg       0.85      0.82      0.83     15600
weighted avg       0.85      0.85      0.85     15600

--- Decision Tree ---
Accuracy: 0.7304

              precision    recall  f1-score   support

           0       0.62      0.63      0.62      5530
           1       0.80      0.78      0.79     10070

    accuracy                           0.73     15600
   macro avg       0.71      0.71      0.71     15600
weighted avg       0.73      0.73      0.73     15600

--- Random Forest ---
Accuracy: 0.8538

              precision    recall  f1-score   support

           0       0.85      0.71      0.77      5530
           1       0.85      0.93      0.89     10070

    accuracy                           0.85     15600
  